# Hamiltonian evolution with fresh GQSP phases

Use the original four Pauli terms on two data qubits, evolution time 22, source scale 0.9999, and Laurent degree 33. The numerical preparation computes the complex Jacobi–Anger coefficients, verifies a spectral complement, and strips the resulting unit vector polynomial into all 67 phase rows. The two-by-two rotations and both controlled walk directions run through FlagQuantum.

Reuse native synth amplitude preparation and the public Pauli SELECT circuit. The two flag qubits and one GQSP signal qubit give an original-sized five-qubit evolution circuit. The 33 inverse selections and 33 forward selections preserve the signed walk powers. Postselect the three flags at zero and apply only the declared scaling before comparing the four complex amplitudes with an independent matrix exponential.

The source used an unseeded input. This workflow uses a reproducible local draw over the same four amplitudes. The recorded instructions count the emitted FlagQuantum IR. Run with FlagQuantum, NumPy, Matplotlib and Jupyter installed.


## Local imports

In [ ]:
from importlib import import_module
from pathlib import Path
from time import perf_counter

from examples.synthesis.algorithms.hamiltonian_simulation.hamiltonian_simulation_with_block_encoding.hamiltonian_simulation_gqsp.overview.main import (
    show_and_save,
    show_factorization,
)
from examples.synthesis.algorithms.hamiltonian_simulation.hamiltonian_simulation_with_block_encoding.hamiltonian_simulation_gqsp.overview.phases import (
    DEGREE,
    factor,
)
from examples.synthesis.algorithms.hamiltonian_simulation.hamiltonian_simulation_with_block_encoding.hamiltonian_simulation_gqsp.overview.workflow import (
    block_projection,
    evolution_projection,
)
from examples.synthesis.algorithms.hamiltonian_simulation.hamiltonian_simulation_with_block_encoding.hamiltonian_simulation_qubitization.overview.model import (
    ALPHA,
    TERMS,
    TIME,
    TOLERANCE,
    fresh_input,
    hamiltonian_matrix,
)


def show(path: Path) -> None:
    viewer = import_module("IPython.display")
    viewer.display(viewer.Image(filename=str(path)))


## Original Hamiltonian

In [ ]:
print("Pauli terms:", TERMS, "evolution time:", TIME, "error target:", TOLERANCE)


In [ ]:
print("Data qubits:", 2, "block flags:", 2, "normalization:", ALPHA)


In [ ]:
initial = fresh_input()
print("Fresh source-sized input:", initial.tolist())


In [ ]:
expected_shape = hamiltonian_matrix().shape
print("Dense reference dimensions:", expected_shape)


## Fresh Laurent coefficients and phase rows

In [ ]:
started = perf_counter()
phases = factor(TIME * ALPHA)
print("Local phase preprocessing seconds:", perf_counter() - started)
show_factorization(phases)


In [ ]:
print("Controlled inverse and forward selections:", DEGREE, DEGREE)


## Verify the original Pauli block encoding

In [ ]:
pauli = block_projection()
print("Pauli block resource width:", pauli.circuit.n_qubits)


In [ ]:
show(show_and_save("pauli", "Pauli block encoding", pauli))


## Execute the five-qubit GQSP program

In [ ]:
evolution = evolution_projection(phases)


In [ ]:
print("Native evolution width:", evolution.circuit.n_qubits)
print(
    "Native evolution circuit IR operations:",
    len(evolution.circuit.to_ir().instructions),
)


In [ ]:
print("Postselection rescaling:", evolution.scale)
print("Projected amplitudes:", evolution.raw.tolist())


In [ ]:
assert evolution.error < TOLERANCE
show(show_and_save("gqsp", "GQSP time evolution", evolution))
